# 27. Prediction and summary
The class notebooks ended with a **Prediction** for a new case (`DT_classifier1.predict(new1_scaled)`) and saved the model (`joblib.dump`). We do the same, and finish with a summary of all results.

* **Needs:** `work/dataset.csv` and the forecast files in `work/`. **Makes:** `work/HAR_model_2023.pkl`.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv
dataset = read_csv(folder + 'work/dataset.csv', index_col=0, parse_dates=True)
har = read_csv(folder + 'work/har_forecasts.csv', index_col=0, parse_dates=True)
garch = read_csv(folder + 'work/garch_forecasts.csv', index_col=0, parse_dates=True)
lstm = read_csv(folder + 'work/lstm_forecasts.csv', index_col=0, parse_dates=True)

## The last forecast of the study
On **31 August 2023**, the last day of the main sample, each model forecast the next week. What happened?

In [3]:
from numpy import sqrt
day = '2023-08-31'
print('realised :', round(sqrt(dataset.loc[day, 'rv5_fwd'] * 252 / 5) * 100, 1), '%')
print('LSTM     :', round(sqrt(lstm.loc[day, 'LSTM'] * 252 / 5) * 100, 1), '%')
print('GARCH    :', round(sqrt(garch.loc[day, 'GARCH'] * 252 / 5) * 100, 1), '%')
print('HAR      :', round(sqrt(har.loc[day, 'HAR'] * 252 / 5) * 100, 1), '%')

realised : 13.3 %
LSTM     : 13.5 %
GARCH    : 14.3 %
HAR      : 15.6 %


## A new situation, and saving the model
The HAR model of the last year (trained up to 31 December 2022):

In [4]:
from numpy import log, exp
from statsmodels.api import OLS, add_constant
data = dataset.loc[:'2023-08-31']
x = log(data[['rv_d', 'rv_w', 'rv_m']].clip(lower=0.00000001))
y = log(data['rv5_fwd'])
train_days = data.loc[:'2022-12-31'].dropna(subset=['rv5_fwd', 'rv_m']).index[:-5]
model = OLS(y.loc[train_days], add_constant(x.loc[train_days]))     # building
best_model = model.fit()                                             # training
smear = exp(best_model.resid).mean()
best_model.params

const   -0.499328
rv_d     0.003528
rv_w     0.172425
rv_m     0.611088
dtype: float64

**Prediction:** suppose that today the ISEQ fell **6%**, after a week of 2% daily moves and a month of 1.2% moves. Next week's expected volatility?

In [5]:
from pandas import DataFrame
new1 = DataFrame([{'const': 1, 'rv_d': log(0.06 ** 2), 'rv_w': log(0.02 ** 2), 'rv_m': log(0.012 ** 2)}])
new_pred = exp(best_model.predict(new1)) * smear
print('forecast for next week (annualised volatility):', round(sqrt(new_pred[0] * 252 / 5) * 100, 1), '%')

forecast for next week (annualised volatility): 22.7 %


In [6]:
import joblib
joblib.dump(best_model, folder + 'work/HAR_model_2023.pkl')
HAR_loaded = joblib.load(folder + 'work/HAR_model_2023.pkl')
HAR_loaded.params.round(3)

const   -0.499
rv_d     0.004
rv_w     0.172
rv_m     0.611
dtype: float64

# Summary of the whole series

| Question | Notebook | Result |
|---|---|---|
| RQ1 / H1 — is the LSTM more accurate than GARCH and HAR? | 18 | **Not supported**: over 2007–2023 the LSTM is significantly *less* accurate (QLIKE 0.463 vs 0.380 and 0.414). Best model in COVID-19 and 2022, worst in the GFC. |
| RQ2 / H2 — do the SHAP fingerprints follow each crisis's origin? | 22 | **Mostly not**: H2a and H2b partly, H2c narrowly supported, H2e not supported. The Irish market supplies about half of every forecast. |
| RQ3 / H3 — does a model trained on past crises cope with a new one? | 19 | **Not as ranked**, but the GFC model (trained only on calm years) was 4.3 × worse than HAR; the COVID model beat HAR. |
| RQ4 / H4 — are the explanations stable across seeds? | 23 | **Not supported**: Spearman 0.7 in the GFC and Irish windows. |
| Robustness — Parkinson measure | 24 | Same conclusions. |
| E1 (exploratory) — SHAP vs spillovers | 25 | Weak agreement (4 of 8, 2 of 5). |
| E2 (exploratory) — calm before the storm? | 26 | Only Germany unusually calm before the GFC. |

Most predictions were not supported — and because they were written down **before** the results (pre-registration), these are genuine findings. Interpreting them, and writing the dissertation, is your work.